# Multi-Tool Research Agent

Day14: arxiv + Google News RSS + LLM. Keyword if/else router degil.


In [ ]:
import arxiv
import feedparser


### Tool 1 arxiv


In [ ]:
def fetch_arxiv_papers(query='hydrogen energy',max_results=3):
    client=arxiv.Client()
    search=arxiv.Search(query=query,max_results=max_results,sort_by=arxiv.SortCriterion.SubmittedDate)
    return [{
        'title':r.title.strip(),
        'summary':r.summary.strip(),
        'url':r.entry_id
    } for r in client.results(search)]

fetch_arxiv_papers()


### Tool 2 Google News


In [ ]:
def fetch_google_news(query='hydrogen energy',max_articles=3):
    url=f'https://news.google.com/rss/search?q={query.replace(" ","+")}+when:7d&hl=en-US&gl=US&ceid=US:en'
    feed=feedparser.parse(url)
    return [{'title':e.title,'link':e.link} for e in feed.entries[:max_articles]]

fetch_google_news()


### Agent


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
from openai import OpenAI
from IPython.display import Markdown

client=OpenAI(base_url='https://openrouter.ai/api/v1',api_key=os.getenv('OPENROUTER_API_KEY'))

def research_agent(query):
    papers=fetch_arxiv_papers(query)
    news=fetch_google_news(query)
    ctx='PAPERS:\n'+'\n'.join(p['title']+' - '+p['summary'][:240] for p in papers)
    ctx+='\nNEWS:\n'+'\n'.join(n['title'] for n in news)
    prompt=f'You are a research agent. Use tools output only.\n## Papers\n## News\n## Takeaway\n\n{ctx}\n\nQuestion: {query}'
    r=client.chat.completions.create(
        model='openai/gpt-oss-120b:free',
        messages=[{'role':'user','content':prompt}]
    )
    return Markdown(r.choices[0].message.content)


In [ ]:
research_agent('hydrogen energy')


### ollama ayni tool ciktisi


In [ ]:
import ollama
papers=fetch_arxiv_papers('human rights',2)
news=fetch_google_news('human rights',2)
ctx='\n'.join(p['title'] for p in papers)+'\n'+'\n'.join(n['title'] for n in news)
r=ollama.chat(model='llama3.2-vision',messages=[{
    'role':'user',
    'content':f'Short research brief from these titles:\n{ctx}'
}])
print(r['message']['content'])


### Sonuc

Iki arac cagirip LLM ozetliyor. Bu multi-tool agent.
